In [11]:
!pip install -q torchaudio==2.4.0 torchvision==0.19 torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 84.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.3/797.3 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 96.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 14.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 MB 5.1 MB/s eta 0:00:0000:0100:01


In [1]:
!mkdir data
!mkdir raft_data
!mkdir saved_explanations
!mkdir saved_models

In [2]:
# %cd .. 
!git clone https://github.com/hoang2306/G-Refer.git

Cloning into 'G-Refer'...
remote: Enumerating objects: 262, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 262 (delta 149), reused 198 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (262/262), 5.50 MiB | 13.61 MiB/s, done.
Resolving deltas: 100% (149/149), done.


In [3]:
!git clone https://github.com/HKUDS/XRec.git

Cloning into 'XRec'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 254 (delta 23), reused 20 (delta 20), pack-reused 224 (from 1)
Receiving objects: 100% (254/254), 187.50 MiB | 28.75 MiB/s, done.
Resolving deltas: 100% (78/78), done.
Updating files: 100% (111/111), done.


In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1v9VNHn4q5dna4c2GWHyC0pgPe5NyBtZl?usp=drive_link

In [4]:
%cd /kaggle/working/

/kaggle/working


In [5]:
import pickle as pkl

dataset_name = 'amazon'
with open(f'XRec/data/{dataset_name}/trn.pkl', 'rb') as f:
    trn = pkl.load(f)

with open(f'XRec/data/{dataset_name}/val.pkl', 'rb') as f:
    val = pkl.load(f)

with open(f'XRec/data/{dataset_name}/tst.pkl', 'rb') as f:
    tst = pkl.load(f)

dict_user = {}
for i in range(len(trn)):
    row = trn.iloc[i]
    uid, iid = row['uid'], row['iid']
    if uid in dict_user:
        dict_user[uid] += 1
    else:
        dict_user[uid] = 1

list_user = sorted(dict_user.items(), key=lambda x: x[1], reverse=True)[:500]
top_user = [data[0] for data in list_user]
trn_filter_user = trn[trn['uid'].isin(top_user)]

dict_item = {}
for i in range(len(trn_filter_user)):
    row = trn.iloc[i]
    uid, iid = row['uid'], row['iid']
    if uid in dict_item:
        dict_item[uid] += 1
    else:
        dict_item[uid] = 1

list_item = sorted(dict_item.items(), key=lambda x: x[1], reverse=True)[400:900]
top_item = [data[0] for data in list_item]
trn_sample = trn_filter_user[trn_filter_user['uid'].isin(top_item)]

dict_user_sample = {}
for i in range(len(trn_sample)):
    row = trn_sample.iloc[i]
    uid, iid = row['uid'], row['iid']
    if iid in dict_user_sample:
        dict_user_sample[iid] += 1
    else:
        dict_user_sample[iid] = 1

list_user_sample = sorted(dict_user_sample.items(), key=lambda x: x[1], reverse=True)

import pandas as pd


def filter_from_trn(df, list_user, list_item):
    
    full_in_trn = df[(df['uid'].isin(list_user)) & (df['iid'].isin(list_item))]
    
    if len(full_in_trn) > 200:
        full_in_trn = full_in_trn.sample(n=200, random_state=42)
    uid_in_trn = df[df['uid'].isin(list_user) & ~df['iid'].isin(list_item)]
    
    if len(uid_in_trn) > 40:
        uid_in_trn = uid_in_trn.sample(n=40, random_state=42)

    iid_in_trn = df[~df['uid'].isin(list_user) & df['iid'].isin(list_item)]
    if len(iid_in_trn) > 40:
        iid_in_trn = iid_in_trn.sample(n=40, random_state=42)
    not_in_trn = df[~df['uid'].isin(list_user) & ~df['iid'].isin(list_item)]
    if len(iid_in_trn) > 20:
        not_in_trn = not_in_trn.sample(n=20, random_state=42)
    sample = pd.concat([full_in_trn, uid_in_trn, iid_in_trn, not_in_trn], ignore_index=True)
    return sample

tst_sample = filter_from_trn(pd.concat([val, tst], ignore_index=True), top_user, top_item)

with open(f'XRec/data/{dataset_name}/trn.pkl', 'wb') as f:
    pkl.dump(trn_sample, f)

with open(f'XRec/data/{dataset_name}/tst.pkl', 'wb') as f:
    pkl.dump(tst_sample, f)

print(f'end of sample process')

end of sample process


In [8]:
!mkdir /kaggle/working/G-Refer/data

mkdir: cannot create directory ‘/kaggle/working/G-Refer/data’: File exists


In [9]:
!cp -r /kaggle/working/XRec/data/amazon /kaggle/working/G-Refer/data/
# !cp -r /kaggle/working/XRec/data/yelp /kaggle/working/G-Refer/data/
# !cp -r /kaggle/working/XRec/data/google /kaggle/working/G-Refer/data/

In [10]:
ls

data/  G-Refer/  raft_data/  saved_explanations/  saved_models/  XRec/


In [82]:
# !cp -r /kaggle/working/XRec/data/amazon /kaggle/working/G-Refer/data/
# !cp -r /kaggle/working/XRec/data/yelp /kaggle/working/G-Refer/data/
# !cp -r /kaggle/working/XRec/data/google /kaggle/working/G-Refer/data/

# print(f'copy raw data from XRec to Grefer')

# !cp /kaggle/working/XRec_sample/amazon/* /kaggle/working/G-Refer/data/amazon
# !cp /kaggle/working/XRec_sample/yelp/* /kaggle/working/G-Refer/data/yelp
# !cp /kaggle/working/XRec_sample/google/* /kaggle/working/G-Refer/data/google


# print(f'copy sample data part from XRec_sample to Grefer')

copy raw data from XRec to Grefer
copy sample data part from XRec_sample to Grefer


In [76]:
# import pickle as pkl 
# import pandas as pd 
# def load_pkl(path):
#     with open(path, 'rb') as f:
#         return pkl.load(f)
        
# def save_pkl(path, data):
#     with open(path, 'wb') as f:
#         pkl.dump(data, f)
#     print(f'saved data to: {path}')
 
# trn = load_pkl(path='/kaggle/working/G-Refer/data/amazon/trn.pkl')
# val = load_pkl(path='/kaggle/working/G-Refer/data/amazon/val.pkl')
# print(len(val) + len(trn))
# trn_df = pd.concat([trn, val]).drop_duplicates()

# # save trn 
# save_pkl(path='/kaggle/working/G-Refer/data/amazon/trn.pkl', data=trn_df)


15232
saved data to: /kaggle/working/G-Refer/data/amazon/trn.pkl


In [13]:
cd /kaggle/working/G-Refer

/kaggle/working/G-Refer


In [14]:
!python code/converter.py --dataset amazon --split trn --text_encoder SentenceBert
# !python code/converter.py --dataset yelp --split trn --text_encoder SentenceBert
# !python code/converter.py --dataset google --split trn --text_encoder SentenceBert

2025-09-08 16:23:44.914512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757348625.461591     144 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757348625.558272     144 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Items with title: 8729
Items without title: 6518
tokenizer_config.json: 100%|███████████████████| 333/333 [00:00<00:00, 1.69MB/s]
vocab.txt: 232kB [00:00, 9.60MB/s]
tokenizer.json: 466kB [00:00, 29.3MB/s]
config.json: 100%|█████████████████████████████| 523/523 [00:00<00:00, 4.28MB/s]
model.safetensors: 100%|██████████████████████| 265M/265M [00:01<00:00, 173MB/s]
Processing texts: 100%|███████████████████████| 240/240 [01:36<00:00, 

In [15]:
!git pull 
!python code/dgl_extractor.py --dataset amazon --split trn 
# !python code/dgl_extractor.py --dataset yelp --split trn 
# !python code/dgl_extractor.py --dataset google --split trn 

Already up to date.
DGL backend not selected or invalid.  Assuming PyTorch for now.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
Processing amazon trn data...
/kaggle/working/G-Refer/code/utils.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serializatio

In [ ]:
!git pull
!python path_retriever/train_linkpred.py --dataset_name amazon --split trn --save_model
# !python path_retriever/train_linkpred.py --dataset_name yelp --split trn --save_model
# !python path_retriever/train_linkpred.py --dataset_name google --split trn --save_model

In [ ]:
# can't run since number of edge very large
# pls use sample data to run pagelink
# sample version consume about 4-5 hours to fully run 
!git pull 
!python path_retriever/pagelink.py --dataset_name amazon --split trn --save_explanation --device_id 0
# !python path_retriever/pagelink.py --dataset_name yelp --split trn --save_explanation --device_id 0
# !python path_retriever/pagelink.py --dataset_name google --split trn --save_explanation --device_id 0

In [19]:
ls /kaggle/input/grefer-reproduce/G-Refer/data/amazon

explanation.json    total.csv      tst.pkl             user_profile.json
item_converter.pkl  total_trn.csv  tst_pred.pkl        val.pkl
item_emb.pkl        total_tst.csv  tst_ref.pkl
item_profile.json   total_val.csv  user_converter.pkl
para_dict.pickle    trn.pkl        user_emb.pkl


In [24]:
ls

cm.sh  ds_inference/  gen_explanations/  path_retriever/   saved_explanations/
code/  ds_training/   images/            README.md         saved_models/
data/  evaluation/    PaGE-Link/         requirements.txt


In [23]:
!mkdir saved_explanations

In [28]:
%cd /kaggle/working/G-Refer/saved_explanations
!gdown 1gSfmwtOHUQIuUrpMR54XFSvyMcZt6dcF
!gdown 1g0q-ZIpdGuiXNwtSxubqV1995IX6RCFc
%cd /kaggle/working/G-Refer

/kaggle/working/G-Refer/saved_explanations
Downloading...
From: https://drive.google.com/uc?id=1gSfmwtOHUQIuUrpMR54XFSvyMcZt6dcF
To: /kaggle/working/G-Refer/saved_explanations/pagelink_amazon_model_trn_pred_edge_to_paths
100%|███████████████████████████████████████| 2.78M/2.78M [00:00<00:00, 179MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1g0q-ZIpdGuiXNwtSxubqV1995IX6RCFc
From (redirected): https://drive.google.com/uc?id=1g0q-ZIpdGuiXNwtSxubqV1995IX6RCFc&confirm=t&uuid=af811158-1076-4df5-b750-36663f59f840
To: /kaggle/working/G-Refer/saved_explanations/pagelink_amazon_model_trn_pred_edge_to_comp_g_edge_mask
100%|████████████████████████████████████████| 239M/239M [00:03<00:00, 60.7MB/s]
/kaggle/working/G-Refer


In [ ]:
# use pagelink output from saved version 
# amazon
!cp -r 

In [21]:
!git pull 
!python code/dense_retriever.py --dataset amazon --split trn 

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 339 bytes | 339.00 KiB/s, done.
From https://github.com/hoang2306/G-Refer
   729cc05..4755a61  main       -> origin/main
Updating 729cc05..4755a61
Fast-forward
 code/dataset.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)
Processing amazon trn data...
/kaggle/working/G-Refer/code/utils.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that co

In [29]:
!git pull
!python code/translation.py --dataset amazon --split trn --k 2

Already up to date.
2025-09-08 16:40:31.285895: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757349631.308607     313 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757349631.315306     313 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/kaggle/working/G-Refer/code/dataset.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the de

In [ ]:
ls /kaggle/input/grefer-reproduce/G-Refer/data

In [ ]:
# ls /kaggle/input/grefer-reproduce/G-Refer/saved_explanations

In [ ]:
# ls /kaggle/input/grefer-reproduce/G-Refer/saved_models

In [ ]:
# ls /kaggle/input/grefer-reproduce/G-Refer/PaGE-Link

In [ ]:
# !cp -r /kaggle/input/grefer-reproduce/G-Refer/saved_explanations /kaggle/working/G-Refer
# !cp -r /kaggle/input/grefer-reproduce/G-Refer/saved_models /kaggle/working/G-Refer
# !cp -r /kaggle/input/grefer-reproduce/G-Refer/data /kaggle/working/G-Refer

# print(f'copy successfully')